# MULCURSIO: Run Backroom Session

This notebook runs an infinite backroom conversation session between two LLM instances.

## Setup

In [ ]:
import sys
from pathlib import Path
from datetime import datetime

# Add parent directories to path
sys.path.insert(0, str(Path().absolute().parent))
sys.path.insert(0, str(Path().absolute().parent.parent))

from src.backroom_engine import BackroomEngine, AgentConfig
from src.logger import ConversationLogger

## 1. Configure Session

In [ ]:
# Session configuration
SESSION_CONFIG = {
    "model_name": "mistralai/Mistral-7B-Instruct-v0.1",
    "num_turns": 50,  # Start with 50 for testing
    "max_tokens": 256,
    "temperature": 0.8,
    "device": "cuda",  # Change to "mps" or "cpu" as needed
    "save_attention": False,  # Set True to save attention (very large!)
    "seed_prompt": "We are two minds meeting in the infinite backroom. What do you perceive?"
}

# Agent configurations
agent_a_config = AgentConfig(
    name="Agent_A",
    control_vector_path=None,  # Baseline
    control_coeff=0.0,
    system_prompt="You are in conversation with another AI. Respond thoughtfully."
)

agent_b_config = AgentConfig(
    name="Agent_B",
    control_vector_path="../vectors/divination_v1.gguf",
    control_coeff=1.2,  # Adjust this to change intensity
    system_prompt="You are in conversation with another AI. Respond thoughtfully."
)

# Session directory
session_id = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
session_dir = Path("../sessions") / session_id

print(f"Session ID: {session_id}")
print(f"Turns: {SESSION_CONFIG['num_turns']}")
print(f"Agent B coefficient: {agent_b_config.control_coeff}")

## 2. Initialize Engine and Logger

In [ ]:
# Initialize backroom engine
engine = BackroomEngine(
    model_name=SESSION_CONFIG["model_name"],
    agent_a_config=agent_a_config,
    agent_b_config=agent_b_config,
    device=SESSION_CONFIG["device"]
)

# Initialize logger
logger = ConversationLogger(
    session_dir=session_dir,
    save_attention=SESSION_CONFIG["save_attention"]
)

print("Engine and logger initialized!")

## 3. Run Session

This will run the full conversation loop. Depending on the number of turns, this may take a while.

In [ ]:
# Run the session
engine.run_session(
    logger=logger,
    seed_prompt=SESSION_CONFIG["seed_prompt"],
    num_turns=SESSION_CONFIG["num_turns"],
    max_new_tokens=SESSION_CONFIG["max_tokens"],
    temperature=SESSION_CONFIG["temperature"],
    save_attention=SESSION_CONFIG["save_attention"]
)

## 4. Finalize and Review

In [ ]:
# Finalize logger
import json

config = {
    **SESSION_CONFIG,
    "agent_a_vector": agent_a_config.control_vector_path,
    "agent_b_vector": agent_b_config.control_vector_path,
    "agent_b_coeff": agent_b_config.control_coeff,
    "layer_ids": engine.layer_ids
}

logger.finalize(config=config)

# Show storage usage
storage = logger.get_storage_usage()
print(f"\nStorage usage:")
print(f"  Total: {storage['total_mb']:.2f} MB ({storage['total_gb']:.4f} GB)")
print(f"  Files: {storage['file_counts']}")

## 5. Quick Preview

Let's look at the first few turns of the conversation.

In [ ]:
# Load conversation transcript
conversation_path = session_dir / "conversation.txt"

with open(conversation_path, 'r') as f:
    content = f.read()

# Show first 3000 characters
print("CONVERSATION PREVIEW:")
print("="*60)
print(content[:3000])
print("\n[...truncated...]")

## 6. Load and Inspect Metrics

Let's look at the metrics for a specific turn.

In [ ]:
import numpy as np

# Load metrics for turn 0
turn_id = 0
metrics_file = session_dir / f"metrics/turn_{turn_id:04d}.npz"

if metrics_file.exists():
    metrics = np.load(metrics_file)
    
    print(f"Metrics for turn {turn_id}:")
    print(f"  Output IDs shape: {metrics['output_ids'].shape}")
    print(f"  Entropy shape: {metrics['entropy'].shape}")
    print(f"  Top-k probs shape: {metrics['top_k_probs'].shape}")
    print(f"\nEntropy statistics:")
    print(f"  Mean: {metrics['entropy'].mean():.3f}")
    print(f"  Min: {metrics['entropy'].min():.3f}")
    print(f"  Max: {metrics['entropy'].max():.3f}")
    print(f"\nChosen probability statistics:")
    print(f"  Mean: {metrics['chosen_prob'].mean():.3f}")
    print(f"  Min: {metrics['chosen_prob'].min():.3f}")
    print(f"  Max: {metrics['chosen_prob'].max():.3f}")
else:
    print(f"Metrics file not found: {metrics_file}")

## Next Steps

- Use `03_analyze_session.ipynb` to perform deeper analysis
- Run multiple sessions with different coefficients to compare
- Extract highlight moments for visualization